In [1]:
import pandas as pd
import ast
import re
import os
from datetime import datetime
from collections import Counter

In [2]:
# ── sklearn hanya dipakai jika keyword mapping tidak cukup ──
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("⚠️  scikit-learn tidak terinstall. Fallback cosine similarity dinonaktifkan.")
    print("    Jalankan: pip install scikit-learn")

In [3]:
INPUT_FILE  = "C:\\Users\\asus3\\Documents\\CPSTNPROJECT\\capstone-ds\\Data\\processed\\linkedin_jobs_20260421_215330_softdev.csv"      # ganti sesuai path file kalian
OUTPUT_DIR  = "C:\\Users\\asus3\\Documents\\CPSTNPROJECT\\capstone-ds\\Data Wrangling\\Data Fixed"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
MIN_JD_LENGTH = 200   # karakter minimum job_description agar dianggap valid

In [4]:
# ── Kategori role standar untuk project ini ──────────────────────
# Sesuaikan dengan scope project: IT roles only
STANDARD_ROLES = {
    "Data Scientist"     : ["data scientist", "ml scientist", "research scientist", "data science"],
    "Data Analyst"       : ["data analyst", "business analyst", "bi analyst", "business intelligence analyst"],
    "Data Engineer"      : ["data engineer", "etl engineer", "data pipeline", "analytics engineer"],
    "ML Engineer"        : ["machine learning engineer", "mlops", "ml engineer", "deep learning engineer", "ai engineer"],
    "BI Developer"       : ["bi developer", "business intelligence", "power bi", "bi development", "tableau developer", "looker"],
    "Backend Developer"  : ["backend developer", "back end developer", "backend engineer", "python developer",
                            "java developer", "golang developer", "api developer", "server side"],
    "Frontend Developer" : ["frontend developer", "front end developer", "frontend engineer", "ui developer",
                            "react developer", "vue developer", "angular developer"],
    "Fullstack Developer": ["fullstack", "full stack", "full-stack"],
    "DevOps Engineer"    : ["devops", "devsecops", "cloud engineer", "sre", "site reliability",
                            "infrastructure engineer", "platform engineer"],
    "Mobile Developer"   : ["mobile developer", "android developer", "ios developer", "flutter developer",
                            "react native", "mobile engineer"],
    "QA Engineer"        : ["qa engineer", "quality assurance", "tester", "test engineer",
                            "qa automation", "software tester"],
    "Software Engineer"  : ["software engineer", "software developer", "programmer", "developer"],  # fallback umum
}

In [5]:
# ── Daftar skill yang diperluas (lebih komprehensif dari scraper) ──
SKILLS_MASTER = {
    # Programming languages
    "python", "java", "javascript", "typescript", "golang", "go", "php", "swift",
    "kotlin", "scala", "r", "c++", "c#", "rust", "dart", "ruby", "bash", "shell",
    # Web frameworks
    "react", "angular", "vue", "nextjs", "nuxtjs", "django", "flask", "fastapi",
    "spring boot", "express", "nestjs", "laravel", "rails", "fiber", "gin",
    # Data & ML
    "tensorflow", "pytorch", "keras", "scikit-learn", "pandas", "numpy", "opencv",
    "huggingface", "transformers", "xgboost", "lightgbm", "catboost", "mlflow",
    "kubeflow", "dvc",
    # Databases
    "sql", "postgresql", "mysql", "mongodb", "redis", "elasticsearch", "cassandra",
    "dynamodb", "firebase", "mariadb", "sqlite", "neo4j", "supabase",
    # Cloud & DevOps
    "aws", "gcp", "azure", "docker", "kubernetes", "k8s", "jenkins", "gitlab ci",
    "github actions", "terraform", "ansible", "prometheus", "grafana", "helm",
    "argocd", "circleci",
    # Big Data
    "hadoop", "spark", "kafka", "airflow", "databricks", "snowflake", "dbt",
    "hive", "flink", "beam",
    # BI Tools
    "tableau", "power bi", "looker", "metabase", "superset", "qlik",
    # Mobile
    "flutter", "android", "ios", "react native", "xamarin",
    # Soft skills & metodologi
    "agile", "scrum", "kanban", "git", "linux", "rest api", "graphql",
    "microservices", "ci/cd", "tdd", "oop",
}

In [6]:
# ════════════════════════════════════════════════════════════════
# STEP 1: LOAD & BASIC CLEANING
# ════════════════════════════════════════════════════════════════
 
def load_and_clean(filepath: str) -> pd.DataFrame:
    print(f"\n{'='*55}")
    print("STEP 1: Load & Basic Cleaning")
    print(f"{'='*55}")
 
    df = pd.read_csv(filepath, encoding="utf-8-sig")
    print(f"  Raw data  : {len(df)} baris")
 
    # Hapus duplikat berdasarkan job_url (paling reliable)
    before = len(df)
    df = df.drop_duplicates(subset=["job_url"], keep="first")
    print(f"  Duplikat  : -{before - len(df)} baris dihapus")
 
    # Hapus baris yang title atau company kosong
    df = df.dropna(subset=["title", "company"])
 
    # Bersihkan whitespace di kolom teks
    for col in ["title", "company", "location", "job_description"]:
        if col in df.columns:
            df[col] = df[col].fillna("").str.strip()
 
    # Filter job_description yang terlalu pendek (tidak informatif)
    before = len(df)
    df = df[df["job_description"].str.len() >= MIN_JD_LENGTH]
    print(f"  JD pendek : -{before - len(df)} baris dihapus (< {MIN_JD_LENGTH} char)")
 
    # Bersihkan HTML tags sisa di job_description
    df["job_description"] = df["job_description"].apply(clean_html)
 
    print(f"  Hasil     : {len(df)} baris bersih")
    return df.reset_index(drop=True)
 
 
def clean_html(text: str) -> str:
    """Hapus HTML tags dan normalisasi whitespace."""
    text = re.sub(r"<[^>]+>", " ", text)           # hapus HTML tags
    text = re.sub(r"&[a-zA-Z]+;", " ", text)        # hapus HTML entities
    text = re.sub(r"\s+", " ", text)                 # normalisasi spasi
    return text.strip()

In [7]:
# STEP 6: SIMPAN OUTPUT
# ════════════════════════════════════════════════════════════════
 
def save_outputs(df: pd.DataFrame):
    print(f"\n{'='*55}")
    print("STEP 6: Simpan Output")
    print(f"{'='*55}")
 
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
 
    # ── A. CSV lengkap (dengan skill vector) untuk AI Engineer ──
    full_path = f"{OUTPUT_DIR}/linkedin_jobs_clean_{ts}.csv"
    df.to_csv(full_path, index=False, encoding="utf-8-sig")
    print(f"  CSV lengkap     → {full_path}")
 
    # ── B. CSV ringkas (tanpa skill vector columns) untuk EDA/review ──
    core_cols = [
        "id", "title", "standardized_role", "company", "location",
        "job_description", "extracted_skills", "skills_count",
        "job_url", "search_role", "scraped_at"
    ]
    slim_path = f"{OUTPUT_DIR}/linkedin_jobs_slim_{ts}.csv"
    df[core_cols].to_csv(slim_path, index=False, encoding="utf-8-sig")
    print(f"  CSV slim        → {slim_path}")
 
    # ── C. Summary JSON untuk AI Engineer ──
    all_skills = []
    for s in df["extracted_skills"]:
        all_skills.extend(s)
    skill_freq = Counter(all_skills)
 
    import json
    summary = {
        "generated_at"        : ts,
        "total_jobs"          : len(df),
        "unique_companies"    : int(df["company"].nunique()),
        "role_distribution"   : df["standardized_role"].value_counts().to_dict(),
        "top_50_skills"       : dict(skill_freq.most_common(50)),
        "avg_skills_per_job"  : round(df["skills_count"].mean(), 2),
        "skills_per_role"     : {},
    }
    for role in df["standardized_role"].unique():
        role_skills = []
        for s in df[df["standardized_role"] == role]["extracted_skills"]:
            role_skills.extend(s)
        top10 = dict(Counter(role_skills).most_common(10))
        summary["skills_per_role"][role] = top10
 
    json_path = f"{OUTPUT_DIR}/data_summary_{ts}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"  Summary JSON    → {json_path}")
 
    return full_path, slim_path


In [8]:
# MAIN
# ════════════════════════════════════════════════════════════════
 
def main():
    print("\n" + "═" * 55)
    print("ETL PIPELINE — DATA PREPROCESSING")
    print("Capstone CC26-PSU404 | Tim Data Scientist")
    print("═" * 55)
 
    # Jalankan pipeline
    df = load_and_clean(INPUT_FILE)
    df = apply_role_standardization(df)
    df = apply_skill_extraction(df)
    df = build_skill_vector(df)
    run_eda(df)
    full_path, slim_path = save_outputs(df)
 
    print("\n" + "═" * 55)
    print("PIPELINE SELESAI")
    print("═" * 55)
    print(f"\nOutput siap diserahkan ke AI Engineer:")
    print(f"  {full_path}")
    print(f"  {slim_path}")
    print(f"\nCatatan untuk AI Engineer:")
    print(f"  - Kolom 'job_description' = bahan baku untuk embedding")
    print(f"  - Kolom 'standardized_role' = label untuk klasifikasi")
    print(f"  - Kolom 'extracted_skills' = list skill per job")
    print(f"  - Kolom 'skill_*' = binary skill vector (baseline)")
 
 
if __name__ == "__main__":
    main()


═══════════════════════════════════════════════════════
ETL PIPELINE — DATA PREPROCESSING
Capstone CC26-PSU404 | Tim Data Scientist
═══════════════════════════════════════════════════════

STEP 1: Load & Basic Cleaning
  Raw data  : 1000 baris
  Duplikat  : -0 baris dihapus
  JD pendek : -2 baris dihapus (< 200 char)
  Hasil     : 998 baris bersih


NameError: name 'apply_role_standardization' is not defined